# URM Sudoku Internal Cycle Visualization

This notebook visualizes how the checkpoint evolves a Sudoku prediction **inside one outer model loop**.

Target command settings:

```bash
python evaluate_trained_model.py \
  --checkpoint checkpoints/URM-sudoku-base \
  --max_problems 4096 \
  --loops 1 \
  --batch_size 4096
```

For this checkpoint, `loops=1` still means the model runs the internal `L_cycles=6` refinement steps once, so we plot the Sudoku map after each of those steps.

In [1]:
import numpy as np
import torch
from IPython.display import HTML, display

from sudoku_trace_utils import (
    build_trace_summary,
    display_trace_summary,
    get_batch,
    load_visualization_setup,
    trace_single_outer_loop,
)

np.set_printoptions(linewidth=140)

In [ ]:
CHECKPOINT = 'checkpoints/URM-sudoku-base'
SPLIT = 'test'
BATCH_SIZE = 4096
BATCH_INDEX = 70
LOOPS = 1
H_CYCLES = None
L_CYCLES = None

setup = load_visualization_setup(
    CHECKPOINT,
    split=SPLIT,
    batch_size=BATCH_SIZE,
    loops=LOOPS,
    h_cycles=H_CYCLES,
    l_cycles=L_CYCLES,
)

batch_info = get_batch(setup['dataloader'], setup['device'], batch_index=BATCH_INDEX)
records = trace_single_outer_loop(setup['model'], batch_info['batch_gpu'])

resolved_h_cycles = setup['model'].model.inner.config.H_cycles
resolved_l_cycles = setup['model'].model.inner.config.L_cycles

print(f"checkpoint step: {setup['step']}")
print(f"set name: {batch_info['set_name']}")
print(f"global batch size: {batch_info['global_batch_size']}")
print(f"outer loops: {setup['model'].model.config.loops}")
print(f"H_cycles: {resolved_h_cycles}")
print(f"L_cycles: {resolved_l_cycles}")
print(f"trace panels: {len(records)}")

Dataset test has 422786 groups.
checkpoint step: 6510
set name: all
global batch size: 4096
outer loops: 1
H_cycles: 1
L_cycles: 6
trace panels: 6


In [3]:
labels = batch_info['batch_cpu']['labels']
valid_mask = (labels != -100).any(dim=1).cpu().numpy()
valid_indices = np.flatnonzero(valid_mask)

final_preds = records[-1].preds
final_exact = ((final_preds == labels) | (labels == -100)).all(dim=1).cpu().numpy() & valid_mask

print(f'valid problems in this batch: {valid_mask.sum()}')
print(f'final exact solved problems: {final_exact.sum()} / {valid_mask.sum()}')
print('first 10 solved sample indices:', valid_indices[final_exact[valid_indices]][:10])
print('first 10 unsolved sample indices:', valid_indices[~final_exact[valid_indices]][:10])

valid problems in this batch: 4096
final exact solved problems: 4027 / 4096
first 10 solved sample indices: [0 1 2 3 4 5 6 7 8 9]
first 10 unsolved sample indices: [166 208 219 236 277 284 312 475 487 504]


In [16]:
SAMPLE_INDEX = 2

summary = build_trace_summary(batch_info['batch_cpu'], records, sample_index=SAMPLE_INDEX)
display_trace_summary(summary, max_cols=4)

'<div style="display:flex;flex-direction:column;gap:16px;"><div style="font-size:22px;font-weight:700;">Sudoku solving trace for sample 2</div><div style="display:grid;grid-template-columns:repeat(4, minmax(220px, 1fr));gap:18px;align-items:start;"><div style="display:flex;flex-direction:column;gap:8px;align-items:center;"><div style="font-size:15px;font-weight:700;text-align:center;white-space:pre-line;">Input</div><table style="border-collapse:collapse;border-spacing:0;"><tr><td style="background:#dbeafe;color:#1d4ed8;font-weight:700;width:34px;height:34px;text-align:center;vertical-align:middle;font-size:20px;line-height:1;border-top:2.5px solid #111827;border-left:2.5px solid #111827">5</td><td style="background:#fffdf7;color:#111827;font-weight:400;width:34px;height:34px;text-align:center;vertical-align:middle;font-size:20px;line-height:1;border-top:2.5px solid #111827;border-left:1px solid #d1d5db"></td><td style="background:#dbeafe;color:#1d4ed8;font-weight:700;width:34px;height

## Notes

- `Original Input` shows the real Sudoku puzzle from the dataset. Blue cells are given clues.
- `Model Input` shows what the evaluation code actually fed into the model. For this checkpoint, `masked_input.enabled=true`, so this is a partially masked version of the solution and usually has many more visible digits.
- `H1/L1` ... `H1/L6` show the predicted Sudoku after each internal refinement step.
- Green cells are correct predictions for blank cells.
- Red cells are incorrect predictions for blank cells.
- Yellow cells are still unresolved or predicted as non-digit tokens.
- `Target` is the ground-truth solved Sudoku.

To inspect a different puzzle, change `SAMPLE_INDEX` and rerun the last cell.